# Regresión logística en 1D con `scikit-learn`

Este cuaderno presenta un ejemplo simple de **clasificación binaria** con una sola característica de entrada.


## 1. Datos de entrenamiento

Cada muestra tiene una sola característica $x$ y una etiqueta binaria $y$.

$$
X = \begin{bmatrix}
1\\
2\\
3\\
4
\end{bmatrix}
$$

$$
y = \begin{bmatrix}
1\\
1\\
0\\
0
\end{bmatrix}
$$

Interpretación rápida:

- las muestras con $x=1$ y $x=2$ pertenecen a la clase $1$,
- las muestras con $x=3$ y $x=4$ pertenecen a la clase $0$.

Como solo existe una característica, el problema puede visualizarse sobre una recta.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, log_loss

# Datos del problema
X = np.array([[1.0],
              [2.0],
              [3.0],
              [4.0]])

y = np.array([1, 1, 0, 0])

# Tabla simple para revisar los datos
datos = pd.DataFrame({
    'x': X.ravel(),
    'y': y
})

datos

## 2. Visualización de los datos

Como el problema es unidimensional, todas las muestras se ubican sobre el eje horizontal.

La coordenada vertical se fija en cero solamente para facilitar la visualización. No representa una segunda característica.


In [ ]:
# Separación de muestras por clase
x_clase_0 = X[y == 0]
x_clase_1 = X[y == 1]

plt.figure(figsize=(8, 4))
plt.scatter(x_clase_0, np.zeros(len(x_clase_0)), label='Clase 0', s=100)
plt.scatter(x_clase_1, np.zeros(len(x_clase_1)), label='Clase 1', s=100)
plt.xlabel('x')
plt.ylabel('posición auxiliar')
plt.title('Datos de entrenamiento')
plt.xlim(0, 5)
plt.ylim(-1, 1)
plt.grid(True)
plt.legend()
plt.show()

## 3. Modelo de regresión logística

La regresión logística no predice directamente una clase. Primero calcula una combinación lineal:

$$
z = b + w x
$$

donde:

- $w$ es el peso asociado a la característica $x$,
- $b$ es el sesgo o intercepto del modelo.

Después, el valor $z$ se transforma en una probabilidad mediante la función sigmoide:

$$
\sigma(z) = \frac{1}{1 + e^{-z}}
$$

Por tanto, la hipótesis del modelo es:

$$
h(x) = \sigma(b + wx)
$$

En clasificación binaria, esta salida se interpreta como la probabilidad de pertenecer a una de las clases. Para este cuaderno se usará como probabilidad de la clase $1$.


In [ ]:
# Función sigmoide para fines explicativos.
# scikit-learn no requiere que la implementemos, pero ayuda a interpretar el modelo.
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z = np.linspace(-8, 8, 300)
sigma_z = sigmoid(z)

plt.figure(figsize=(8, 4))
plt.plot(z, sigma_z)
plt.axhline(0.5, linestyle='--', linewidth=1, label='Umbral 0.5')
plt.axvline(0, linestyle='--', linewidth=1)
plt.xlabel('z')
plt.ylabel(r'$\sigma(z)$')
plt.title('Función sigmoide')
plt.grid(True)
plt.legend()
plt.show()

## 4. Entrenamiento con `scikit-learn`

En lugar de programar manualmente el descenso por gradiente, se utilizará `LogisticRegression`.

El flujo básico de trabajo en `scikit-learn` es:

1. crear el modelo,
2. entrenarlo con `.fit(X, y)`,
3. obtener probabilidades con `.predict_proba(X)`,
4. obtener clases con `.predict(X)`.

En este ejemplo se usa el solucionador `lbfgs`, que es una opción estándar para regresión logística.

Además, `scikit-learn` aplica regularización por defecto. La regularización ayuda a controlar el tamaño de los coeficientes y evita modelos excesivamente extremos. El parámetro `C` controla la regularización de forma inversa: valores pequeños de `C` implican regularización más fuerte.


In [ ]:
# Creación del modelo
# C=1.0 corresponde al valor por defecto y mantiene regularización L2.
modelo = LogisticRegression(solver='lbfgs', C=1.0, fit_intercept=True)

# Entrenamiento
modelo.fit(X, y)

# Parámetros aprendidos
b = modelo.intercept_[0]
w = modelo.coef_[0, 0]

print(f'Intercepto b = {b:.6f}')
print(f'Peso w       = {w:.6f}')
print(f'Clases del modelo = {modelo.classes_}')

## 5. Interpretación de los parámetros

El modelo entrenado tiene la forma:

$$
z = b + wx
$$

Como el peso $w$ resulta negativo en este conjunto de datos, la probabilidad de la clase $1$ disminuye cuando $x$ aumenta.

Esto coincide con los datos de entrenamiento:

- valores pequeños de $x$ pertenecen a la clase $1$,
- valores grandes de $x$ pertenecen a la clase $0$.

La probabilidad estimada para la clase $1$ puede calcularse con:

$$
P(y=1 \mid x) = \sigma(b + wx)
$$


In [ ]:
# Cálculo manual de z y de la probabilidad de la clase 1
z_manual = b + w * X.ravel()
prob_manual = sigmoid(z_manual)

# Probabilidades calculadas por scikit-learn
prob_sklearn = modelo.predict_proba(X)

# Ubicamos la columna correspondiente a la clase 1
indice_clase_1 = np.where(modelo.classes_ == 1)[0][0]
prob_clase_1 = prob_sklearn[:, indice_clase_1]

comparacion = pd.DataFrame({
    'x': X.ravel(),
    'z = b + wx': z_manual,
    'sigmoid(z)': prob_manual,
    'predict_proba clase 1': prob_clase_1
})

comparacion

## 6. Predicción de clases

La regresión logística entrega probabilidades. Para convertirlas en clases, se aplica una regla de decisión.

La regla más común para clasificación binaria es usar el umbral $0.5$:

$$
\hat{y} =
\begin{cases}
1, & \text{si } P(y=1 \mid x) \geq 0.5\\
0, & \text{si } P(y=1 \mid x) < 0.5
\end{cases}
$$

En `scikit-learn`, esta regla se aplica directamente con el método `.predict(X)`.


In [ ]:
# Predicción de clases
predicciones = modelo.predict(X)

# Exactitud
exactitud = accuracy_score(y, predicciones)

resultados = pd.DataFrame({
    'x': X.ravel(),
    'y_real': y,
    'probabilidad_clase_1': prob_clase_1,
    'y_predicha': predicciones
})

print(f'Exactitud del modelo: {100 * exactitud:.2f}%')
resultados

## 7. Función de costo: entropía cruzada

La regresión logística se entrena minimizando una función de pérdida asociada a la clasificación binaria.

Para una muestra individual, la pérdida se puede escribir como:

$$
\ell(y, h(x)) = -\left[y \log(h(x)) + (1-y)\log(1-h(x))\right]
$$

Para $m$ muestras, la pérdida promedio es:

$$
J(w,b) = -\frac{1}{m}\sum_{i=1}^{m}\left[
y^{(i)}\log(h(x^{(i)})) +
(1-y^{(i)})\log(1-h(x^{(i)}))
\right]
$$

En `scikit-learn`, el ajuste del modelo se realiza internamente mediante algoritmos de optimización. Por eso no se necesita escribir el ciclo de entrenamiento manualmente.


In [ ]:
# Cálculo de la pérdida logarítmica sobre los datos de entrenamiento
# Se usa la implementación de scikit-learn para evitar errores numéricos.
loss = log_loss(y, prob_sklearn, labels=modelo.classes_)

print(f'Log loss sobre entrenamiento: {loss:.6f}')

## 8. Frontera de decisión en 1D

La frontera de decisión ocurre cuando el modelo es indiferente entre las dos clases.

Esto sucede cuando:

$$
P(y=1 \mid x) = 0.5
$$

Como la sigmoide vale $0.5$ cuando su argumento es cero, se cumple:

$$
b + wx = 0
$$

Despejando $x$:

$$
x_{\text{frontera}} = -\frac{b}{w}
$$

En una dimensión, la frontera no es una recta ni una curva: es un único punto sobre el eje $x$.


In [ ]:
# Frontera de decisión
x_boundary = -b / w

print(f'Frontera de decisión: x = {x_boundary:.6f}')

## 9. Curva de probabilidad y frontera de decisión

La siguiente gráfica muestra:

- los datos reales,
- la probabilidad estimada de pertenecer a la clase $1$,
- el umbral de decisión $0.5$,
- la frontera de decisión.

Esta visualización es útil porque conecta la interpretación geométrica con la interpretación probabilística del modelo.


In [ ]:
# Puntos para graficar la curva de probabilidad
x_grid = np.linspace(0, 5, 400).reshape(-1, 1)
prob_grid = modelo.predict_proba(x_grid)[:, indice_clase_1]

plt.figure(figsize=(9, 5))
plt.plot(x_grid.ravel(), prob_grid, label='P(y=1 | x)')
plt.scatter(X.ravel(), y, s=100, label='Datos reales')
plt.axhline(0.5, linestyle='--', linewidth=1, label='Umbral 0.5')
plt.axvline(x_boundary, linestyle='--', linewidth=2, label='Frontera de decisión')
plt.xlabel('x')
plt.ylabel('Probabilidad / clase')
plt.title('Regresión logística ajustada con scikit-learn')
plt.xlim(0, 5)
plt.ylim(-0.05, 1.05)
plt.grid(True)
plt.legend()
plt.show()

## 10. Matriz de confusión

La matriz de confusión resume los aciertos y errores del modelo.

Para clasificación binaria, compara:

- clase real $0$ frente a clase predicha $0$,
- clase real $0$ frente a clase predicha $1$,
- clase real $1$ frente a clase predicha $0$,
- clase real $1$ frente a clase predicha $1$.

En este conjunto de datos pequeño, la matriz permite verificar rápidamente si el modelo clasificó correctamente las cuatro muestras.


In [ ]:
cm = confusion_matrix(y, predicciones, labels=modelo.classes_)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=modelo.classes_)
disp.plot()
plt.title('Matriz de confusión')
plt.show()

## 11. Efecto de la regularización

El parámetro `C` controla la intensidad de la regularización de forma inversa:

- `C` pequeño: regularización más fuerte,
- `C` grande: regularización más débil.

La regularización no suele cambiar la idea general del modelo, pero sí puede modificar la inclinación de la curva sigmoide y la magnitud de los coeficientes.


In [ ]:
C_values = [0.1, 1.0, 10.0, 100.0]
resumen_C = []

plt.figure(figsize=(9, 5))

for C in C_values:
    modelo_C = LogisticRegression(solver='lbfgs', C=C, fit_intercept=True)
    modelo_C.fit(X, y)
    idx_1 = np.where(modelo_C.classes_ == 1)[0][0]
    prob_C = modelo_C.predict_proba(x_grid)[:, idx_1]
    b_C = modelo_C.intercept_[0]
    w_C = modelo_C.coef_[0, 0]
    frontera_C = -b_C / w_C

    resumen_C.append({
        'C': C,
        'b': b_C,
        'w': w_C,
        'frontera': frontera_C
    })

    plt.plot(x_grid.ravel(), prob_C, label=f'C={C}')

plt.scatter(X.ravel(), y, s=100, label='Datos reales')
plt.axhline(0.5, linestyle='--', linewidth=1, label='Umbral 0.5')
plt.xlabel('x')
plt.ylabel('P(y=1 | x)')
plt.title('Efecto de C sobre la curva logística')
plt.xlim(0, 5)
plt.ylim(-0.05, 1.05)
plt.grid(True)
plt.legend()
plt.show()

pd.DataFrame(resumen_C)

## Ejercicios propuestos

1. Cambie el valor de `C` y observe cómo cambia la curva de probabilidad.
2. Agregue más puntos de entrenamiento y vuelva a ajustar el modelo.
3. Cambie el umbral de decisión de $0.5$ a $0.6$ o $0.7$ y observe cómo cambian las predicciones.
4. Modifique las etiquetas para crear un caso donde el modelo cometa errores.
5. Extienda el ejemplo a dos características y grafique una frontera de decisión en 2D.
